# Campus Placement Data Analysis Using Python
### Fundamentals of Data Science | Beginner-Level Semester Project

**Project theme:** Exploring campus placement patterns using Python, Pandas, NumPy, Matplotlib and Seaborn.

**Methods covered:** Data inspection, data cleaning, descriptive statistics, probability, frequency distribution, data visualisation, covariance, correlation, hypothesis testing, Logistic Regression and simple linear regression.

> **Dataset note:** This project uses a mock campus-placement dataset. Findings describe only the records supplied with the project and must not be generalised to every college or employer.

## 1. Introduction

Campus placement outcomes can be associated with several student attributes, such as academic performance, aptitude scores, projects and internships. This project uses a student placement dataset to practise the basic stages of a data science workflow.

### Problem statement
To examine the structure and quality of a campus placement dataset, summarise important student attributes, visualise placement patterns and study the relationship between CGPA and package using simple linear regression.

### Objectives
1. Load and inspect a CSV dataset.
2. Identify missing values, duplicate records and inconsistent entries.
3. Clean selected columns using basic Pandas operations.
4. Calculate descriptive statistics and simple probabilities.
5. Compare placement outcomes across branches.
6. Create visualisations to communicate patterns.
7. Calculate covariance and correlation.
8. Apply an independent-samples t-test to compare CGPA between placement groups.
9. Apply a chi-square test of independence to examine the relationship between branch and placement status.
10. Apply Logistic Regression using CGPA to classify placement status.
11. Fit and interpret a simple linear regression model using CGPA as the independent variable and package as the dependent variable.

### Scope
The project is intentionally limited to beginner-level FDS concepts. It includes two basic hypothesis tests, binary classification using Logistic Regression, and simple linear regression. It does not use clustering, multiple regression or advanced machine-learning algorithms.

## 2. Tools and Libraries

- **Python:** Programming language
- **Pandas:** Reading, inspecting and cleaning tabular data
- **NumPy:** Numerical calculations
- **Matplotlib:** Basic charts
- **Seaborn:** Statistical visualisations
- **Scikit-learn:** Training and evaluating the Logistic Regression model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

## 3. Load the Dataset

Place `mock_placement_raw.csv` in the same folder as this notebook. In Google Colab, use the upload option below. If the file has a different name, select that file instead.

In [ ]:
# Google Colab users can upload the CSV using this cell.
# If you are running locally, keep the CSV in the notebook folder
# and enter its file name when asked.

try:
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
except ImportError:
    file_name = input("Enter CSV file name (example: mock_placement_raw.csv): ")

df = pd.read_csv(file_name)
df.columns = df.columns.str.strip()

print("Dataset loaded successfully.")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
display(df.head())

## 4. Understand the Dataset

Before cleaning, inspect the column names, data types, first records, last records and summary statistics. This helps us understand what information is available.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nDataset information:")
df.info()

print("\\nFirst five records:")
display(df.head())

print("\\nSummary of numeric columns:")
display(df.describe(include="all").T)

## 5. Check Data Quality

We check missing values, duplicate student IDs and the values recorded in the placement column. A missing placement label is not automatically treated as 'No'.

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum().sort_values(ascending=False))

if "Student ID" in df.columns:
    print("Duplicate Student IDs:", df["Student ID"].duplicated().sum())

if "placed" in df.columns:
    print("Placement labels before cleaning:")
    display(df["placed"].value_counts(dropna=False))

if "branch" in df.columns:
    print("Branch labels before cleaning:")
    display(df["branch"].value_counts(dropna=False))

## 6. Data Cleaning and Preprocessing

The cleaning steps are deliberately simple:

- Work on a copy so the original dataset remains unchanged.
- Remove duplicate Student IDs, keeping the first record.
- Convert common placeholders such as `?` and `NA` into missing values.
- Convert selected numeric columns into numeric data types.
- Standardise Yes/No placement labels.
- Remove extra spaces and standardise branch labels.
- Mark values outside clear scale limits as missing (for example, CGPA above 10 or communication score above 10).

We do not remove unusual but potentially valid package values just because they are large. Such values should be investigated rather than automatically deleted.

In [ ]:
data = df.copy()

# Replace common text placeholders with missing values.
data = data.replace(["?", " ?", "NA", "N/A", "", " "], np.nan)

# Remove duplicate student records when a Student ID column is available.
if "Student ID" in data.columns:
    rows_before = len(data)
    data = data.drop_duplicates(subset="Student ID", keep="first")
    print("Duplicate records removed:", rows_before - len(data))

# Convert relevant columns to numbers.
numeric_columns = [
    "CGPA", "active_backlogs", "aptitude_score", "GATE Score",
    "gate_rank", "num_projects", "ai_ml_projects", "internships",
    "certifications", "dsa_problems_solved", "leetcode_rating",
    "communication_score", "Package (LPA)"
]

for column in numeric_columns:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

# Standardise placement labels.
if "placed" in data.columns:
    data["placed"] = data["placed"].astype("string").str.strip().str.lower()
    data["placed"] = data["placed"].replace({
        "yes": "Yes", "y": "Yes", "1": "Yes", "true": "Yes",
        "no": "No", "n": "No", "0": "No", "false": "No"
    })

# Standardise branch labels and merge common CSE spellings.
if "branch" in data.columns:
    data["branch"] = data["branch"].astype("string").str.strip().str.upper()
    branch_map = {
        "C.S.E": "CSE",
        "COMPUTER SCIENCE": "CSE",
        "COMPUTER SCIENCE AND ENGINEERING": "CSE",
        "AI&DS": "AI&DS",
        "AIDS": "AI&DS"
    }
    data["branch"] = data["branch"].replace(branch_map)

# Standardise selected text columns.
for column in ["primary_language", "gate_appeared", "gate_qualified",
               "active_github", "company_type", "role"]:
    if column in data.columns:
        data[column] = data[column].astype("string").str.strip()

# Apply clear validity checks.
if "CGPA" in data.columns:
    data.loc[(data["CGPA"] < 0) | (data["CGPA"] > 10), "CGPA"] = np.nan

if "Package (LPA)" in data.columns:
    data.loc[data["Package (LPA)"] < 0, "Package (LPA)"] = np.nan

if "communication_score" in data.columns:
    data.loc[(data["communication_score"] < 0) |
             (data["communication_score"] > 10), "communication_score"] = np.nan

if "aptitude_score" in data.columns:
    data.loc[(data["aptitude_score"] < 0) |
             (data["aptitude_score"] > 100), "aptitude_score"] = np.nan

print("Cleaning completed.")
print("Rows after cleaning:", len(data))
print("\\nMissing values after cleaning:")
display(data.isnull().sum().sort_values(ascending=False))

## 7. Review the Cleaned Dataset

Compare the raw and cleaned data sizes and inspect the cleaned records. This gives a simple view of the effect of preprocessing.

In [ ]:
print("Raw dataset shape:", df.shape)
print("Cleaned dataset shape:", data.shape)

display(data.head())

if "placed" in data.columns:
    print("Placement labels after cleaning:")
    display(data["placed"].value_counts(dropna=False))

## 8. Separate Known Placement Outcomes

Only records with a known placement label are used for placement probability and branch-wise placement analysis. Unknown labels remain excluded from these calculations.

In [ ]:
if "placed" not in data.columns:
    raise ValueError("The dataset must contain a 'placed' column.")

known_placement = data[data["placed"].isin(["Yes", "No"])].copy()
placed_students = known_placement[known_placement["placed"] == "Yes"].copy()
not_placed_students = known_placement[known_placement["placed"] == "No"].copy()

print("Students with known placement status:", len(known_placement))
print("Placed students:", len(placed_students))
print("Not-placed students:", len(not_placed_students))
print("Records with unknown placement status:", data["placed"].isna().sum())

## 9. Descriptive Statistics

We calculate mean, median, mode, sample variance and sample standard deviation using their mathematical formulas and simple loops. We avoid ready-made statistical methods such as `mean()`, `median()`, `mode()`, `var()` and `std()` for these calculations.

The formulas are:

- Mean = Sum of observations / Number of observations
- Median = Middle value after sorting the observations
- Mode = Most frequently occurring value
- Sample variance = Sum of squared deviations from mean / (n - 1)
- Sample standard deviation = Square root of sample variance

The `describe()` table is retained only as an additional dataset summary; the main CGPA calculations are performed manually.

In [ ]:
# Descriptive statistics calculated using mathematical formulas.
# We are not using Pandas mean(), median(), mode(), var() or std() here.

cgpa_values = []
for value in data["CGPA"]:
    if pd.notna(value):
        cgpa_values.append(float(value))

if len(cgpa_values) > 0:
    # 1. Mean = Sum of all observations / Number of observations
    total = 0
    for value in cgpa_values:
        total = total + value
    cgpa_mean = total / len(cgpa_values)

    # 2. Median = Middle value after arranging observations in ascending order
    sorted_cgpa = cgpa_values.copy()
    sorted_cgpa.sort()
    n = len(sorted_cgpa)

    if n % 2 == 1:
        cgpa_median = sorted_cgpa[n // 2]
    else:
        middle1 = sorted_cgpa[(n // 2) - 1]
        middle2 = sorted_cgpa[n // 2]
        cgpa_median = (middle1 + middle2) / 2

    # 3. Mode = Value occurring the maximum number of times
    frequency = {}
    for value in cgpa_values:
        if value in frequency:
            frequency[value] = frequency[value] + 1
        else:
            frequency[value] = 1

    highest_frequency = 0
    cgpa_mode = None
    for value in frequency:
        if frequency[value] > highest_frequency:
            highest_frequency = frequency[value]
            cgpa_mode = value

    # 4. Sample variance = Sum of squared deviations / (n - 1)
    squared_deviation_total = 0
    for value in cgpa_values:
        difference = value - cgpa_mean
        squared_deviation_total = squared_deviation_total + difference ** 2

    if n > 1:
        cgpa_variance = squared_deviation_total / (n - 1)
        # 5. Standard deviation = Square root of variance
        cgpa_std = cgpa_variance ** 0.5
    else:
        cgpa_variance = 0
        cgpa_std = 0

    print("Number of CGPA observations:", n)
    print("CGPA mean:", round(cgpa_mean, 2))
    print("CGPA median:", round(cgpa_median, 2))
    print("CGPA mode:", round(cgpa_mode, 2))
    print("CGPA sample variance:", round(cgpa_variance, 2))
    print("CGPA sample standard deviation:", round(cgpa_std, 2))
else:
    print("No valid CGPA values are available.")

# Show a descriptive summary table for the other selected columns.
# This table is for display; the CGPA calculations above use formulas.
statistics_columns = [
    column for column in [
        "CGPA", "aptitude_score", "communication_score",
        "num_projects", "internships", "dsa_problems_solved"
    ] if column in data.columns
]
display(data[statistics_columns].describe().T)

## 10. Basic Probability

For this project, placement probability is calculated using records with a known placement outcome.

\[
P(Placed)=\frac{\text{Number of placed students}}{\text{Students with known status}}
\]

\[
P(Not\ placed)=\frac{\text{Number of not-placed students}}{\text{Students with known status}}
\]

In [ ]:
# Probability calculated using the formula:
# Probability = Number of favourable outcomes / Total number of outcomes

total_known = 0
placed_count = 0
not_placed_count = 0

for status in data["placed"]:
    if status == "Yes":
        placed_count = placed_count + 1
        total_known = total_known + 1
    elif status == "No":
        not_placed_count = not_placed_count + 1
        total_known = total_known + 1

if total_known > 0:
    probability_placed = placed_count / total_known
    probability_not_placed = not_placed_count / total_known

    print("Total students with known status:", total_known)
    print("Placed students:", placed_count)
    print("Not-placed students:", not_placed_count)
    print("P(Placed) = Placed students / Total known students")
    print("P(Placed) =", placed_count, "/", total_known, "=", round(probability_placed, 3))
    print("P(Not placed) = Not-placed students / Total known students")
    print("P(Not placed) =", not_placed_count, "/", total_known, "=", round(probability_not_placed, 3))
    print("Sum of probabilities =", round(probability_placed + probability_not_placed, 3))
else:
    print("No records with known placement status are available.")

## 11. Frequency Distribution by Branch

A frequency table counts the number of placed and not-placed students in each branch. We also calculate row-wise percentages because branch sizes may differ.

In [ ]:
if "branch" in known_placement.columns and len(known_placement) > 0:
    branch_table = pd.crosstab(known_placement["branch"], known_placement["placed"])
    display(branch_table)

    branch_percentage = pd.crosstab(
        known_placement["branch"],
        known_placement["placed"],
        normalize="index"
    ) * 100

    print("Placement percentage within each branch:")
    display(branch_percentage.round(1))
else:
    print("Branch information is not available.")

## 12. Visualisation: Placement Distribution

The bar chart compares the number of students recorded as placed and not placed.

In [ ]:
if len(known_placement) > 0:
    placement_counts = known_placement["placed"].value_counts().reindex(["Yes", "No"], fill_value=0)
    placement_counts.plot(kind="bar", color=["#4C9F70", "#D98282"])
    plt.title("Placement Status Distribution")
    plt.xlabel("Placement status")
    plt.ylabel("Number of students")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 13. Visualisation: Branch-wise Placement

This grouped bar chart compares placement counts across branches. Use the percentage table in the previous section as well, because counts are affected by branch size.

In [ ]:
if "branch" in known_placement.columns and len(known_placement) > 0:
    branch_table.plot(kind="bar", figsize=(10, 5))
    plt.title("Placement Status by Branch")
    plt.xlabel("Branch")
    plt.ylabel("Number of students")
    plt.xticks(rotation=35, ha="right")
    plt.legend(title="Placement status")
    plt.tight_layout()
    plt.show()

## 14. Visualisation: CGPA Distribution

A histogram shows how CGPA values are distributed among the students.

In [ ]:
if "CGPA" in data.columns:
    plt.hist(data["CGPA"].dropna(), bins=10, edgecolor="black")
    plt.title("Distribution of Student CGPA")
    plt.xlabel("CGPA")
    plt.ylabel("Number of students")
    plt.tight_layout()
    plt.show()

## 15. Visualisation: CGPA Boxplot

A boxplot shows the median, quartiles and possible unusually high or low values.

In [ ]:
if "CGPA" in data.columns and data["CGPA"].notna().any():
    sns.boxplot(x=data["CGPA"], color="#83B6C9")
    plt.title("Boxplot of CGPA")
    plt.xlabel("CGPA")
    plt.tight_layout()
    plt.show()

## 16. Visualisation: Aptitude Score

This chart compares aptitude-score distributions for students with known placement outcomes. It is useful for visual exploration but does not establish cause and effect.

In [ ]:
if {"aptitude_score", "placed"}.issubset(known_placement.columns):
    sns.histplot(
        data=known_placement,
        x="aptitude_score",
        hue="placed",
        bins=12,
        element="step",
        stat="count",
        common_norm=False
    )
    plt.title("Aptitude Score by Placement Status")
    plt.xlabel("Aptitude score")
    plt.tight_layout()
    plt.show()

## 17. Covariance and Correlation

We calculate covariance and Pearson correlation manually using mathematical formulas rather than Pandas `.cov()` and `.corr()`.

Sample covariance:

\[
Cov(X,Y)=\frac{\sum (X-\bar X)(Y-\bar Y)}{n-1}
\]

Pearson correlation:

\[
r=\frac{\sum (X-\bar X)(Y-\bar Y)}
{\sqrt{\sum(X-\bar X)^2\sum(Y-\bar Y)^2}}
\]

A positive correlation means the variables tend to increase together. A negative correlation means one tends to decrease as the other increases. A value near zero indicates a weak linear relationship.

Correlation does not prove that one variable causes another.

In [ ]:
# Covariance and correlation calculated with their mathematical formulas.
# Sample covariance:
# Cov(X,Y) = Sum[(X - mean_X)(Y - mean_Y)] / (n - 1)
#
# Pearson correlation:
# r = Cov(X,Y) / (standard_deviation_X * standard_deviation_Y)

relationship_columns = [
    column for column in [
        "CGPA", "aptitude_score", "communication_score",
        "dsa_problems_solved", "Package (LPA)"
    ] if column in data.columns
]

covariance_values = {}
correlation_values = {}

for column_x in relationship_columns:
    covariance_values[column_x] = {}
    correlation_values[column_x] = {}

    for column_y in relationship_columns:
        paired_x = []
        paired_y = []

        for row_index in range(len(data)):
            value_x = data.iloc[row_index][column_x]
            value_y = data.iloc[row_index][column_y]

            if pd.notna(value_x) and pd.notna(value_y):
                paired_x.append(float(value_x))
                paired_y.append(float(value_y))

        n = len(paired_x)

        if n < 2:
            covariance_values[column_x][column_y] = np.nan
            correlation_values[column_x][column_y] = np.nan
        else:
            total_x = 0
            total_y = 0
            for i in range(n):
                total_x = total_x + paired_x[i]
                total_y = total_y + paired_y[i]

            mean_x = total_x / n
            mean_y = total_y / n

            cross_product_total = 0
            squared_x_total = 0
            squared_y_total = 0

            for i in range(n):
                difference_x = paired_x[i] - mean_x
                difference_y = paired_y[i] - mean_y

                cross_product_total = cross_product_total + (difference_x * difference_y)
                squared_x_total = squared_x_total + difference_x ** 2
                squared_y_total = squared_y_total + difference_y ** 2

            covariance = cross_product_total / (n - 1)

            if squared_x_total == 0 or squared_y_total == 0:
                correlation = np.nan
            else:
                correlation = cross_product_total / ((squared_x_total * squared_y_total) ** 0.5)

            covariance_values[column_x][column_y] = covariance
            correlation_values[column_x][column_y] = correlation

covariance_table = pd.DataFrame(covariance_values).T
correlation_table = pd.DataFrame(correlation_values).T

print("Covariance matrix calculated using the formula:")
display(covariance_table.round(2))

print("Correlation matrix calculated using Pearson's formula:")
display(correlation_table.round(2))

## 18. Correlation Heatmap

The heatmap provides a visual summary of pairwise correlations. Focus on the direction and approximate strength of the values.

In [ ]:
# Display the correlation values calculated manually in the previous section.
correlation_matrix = correlation_table

if not correlation_matrix.empty:
    sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm",
                vmin=-1, vmax=1, fmt=".2f")
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()

## 19. Independent-Samples t-Test: CGPA and Placement Status

This test compares the average CGPA of two separate groups:

- Group 1: students marked **Placed**
- Group 2: students marked **Not Placed**

We use **Welch's independent-samples t-test**. It is suitable when the two groups may have different sample sizes or variances.

**Hypotheses**

- \(H_0\): The population mean CGPA is equal for placed and not-placed students.
- \(H_1\): The population mean CGPA is different for the two groups.

**Significance level:** \(\alpha = 0.05\) (5%).

**Formula**

\[
t=\frac{\bar X_1-\bar X_2}
{\sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}}
\]

Welch's degrees of freedom are calculated using the Welch–Satterthwaite formula. The code below calculates the group means, sample variances, t-statistic and degrees of freedom using loops and mathematical formulas. It compares the result with a 5% two-tailed critical-value table; no ready-made t-test function is used.

**Decision rule:** If \(|t|\) is greater than the critical t-value, reject \(H_0\). Otherwise, do not reject \(H_0\).

> This test checks for a difference in average CGPA. It does not prove that CGPA causes placement.

In [ ]:
# Independent-samples Welch t-test calculated manually.
# Group 1 = Placed students; Group 2 = Not-placed students.

placed_cgpa = []
not_placed_cgpa = []

for value in placed_students["CGPA"]:
    if pd.notna(value):
        placed_cgpa.append(float(value))

for value in not_placed_students["CGPA"]:
    if pd.notna(value):
        not_placed_cgpa.append(float(value))

n1 = len(placed_cgpa)
n2 = len(not_placed_cgpa)

if n1 < 2 or n2 < 2:
    print("At least two valid CGPA values are required in each group.")
else:
    # Calculate group means manually.
    mean1 = 0
    for value in placed_cgpa:
        mean1 = mean1 + value
    mean1 = mean1 / n1

    mean2 = 0
    for value in not_placed_cgpa:
        mean2 = mean2 + value
    mean2 = mean2 / n2

    # Calculate sample variances manually.
    squared_deviation1 = 0
    for value in placed_cgpa:
        squared_deviation1 = squared_deviation1 + (value - mean1) ** 2
    variance1 = squared_deviation1 / (n1 - 1)

    squared_deviation2 = 0
    for value in not_placed_cgpa:
        squared_deviation2 = squared_deviation2 + (value - mean2) ** 2
    variance2 = squared_deviation2 / (n2 - 1)

    # Welch t-statistic.
    standard_error = ((variance1 / n1) + (variance2 / n2)) ** 0.5

    if standard_error == 0:
        print("The t-test cannot be calculated because the standard error is zero.")
    else:
        t_statistic = (mean1 - mean2) / standard_error

        # Welch-Satterthwaite degrees of freedom.
        numerator_df = ((variance1 / n1) + (variance2 / n2)) ** 2
        denominator_df = (
            ((variance1 / n1) ** 2) / (n1 - 1)
            + ((variance2 / n2) ** 2) / (n2 - 1)
        )
        degrees_freedom = numerator_df / denominator_df

        # 5% two-tailed t critical values.
        # For an in-between df, use the next lower table df.
        t_table = {
            1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571,
            6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262, 10: 2.228,
            12: 2.179, 15: 2.131, 20: 2.086, 25: 2.060,
            30: 2.042, 40: 2.021, 60: 2.000, 120: 1.980
        }

        table_df = 1
        for df_value in t_table:
            if df_value <= degrees_freedom and df_value > table_df:
                table_df = df_value

        critical_t = t_table[table_df]

        print("INDEPENDENT-SAMPLES t-TEST (WELCH)")
        print("-----------------------------------")
        print("Placed group count:", n1)
        print("Not-placed group count:", n2)
        print("Mean CGPA of placed students:", round(mean1, 3))
        print("Mean CGPA of not-placed students:", round(mean2, 3))
        print("Sample variance of placed group:", round(variance1, 3))
        print("Sample variance of not-placed group:", round(variance2, 3))
        print("Calculated t-statistic:", round(t_statistic, 3))
        print("Welch degrees of freedom:", round(degrees_freedom, 2))
        print("Critical t-value used (5%, two-tailed):", critical_t)
        print("Table degrees of freedom used:", table_df)

        if abs(t_statistic) > critical_t:
            print("Decision: Reject H0.")
            print("Conclusion: The sample provides evidence of a difference in average CGPA between the two placement groups.")
        else:
            print("Decision: Do not reject H0.")
            print("Conclusion: The sample does not provide sufficient evidence of a difference in average CGPA between the two placement groups.")

        print("Note: The critical value is taken from a t-table, using the next lower listed degrees of freedom.")

## 20. Chi-Square Test of Independence: Branch and Placement Status

This test checks whether **branch** and **placement status** are associated in the dataset.

**Hypotheses**

- \(H_0\): Branch and placement status are independent.
- \(H_1\): Branch and placement status are associated.

**Significance level:** \(\alpha = 0.05\) (5%).

**Expected frequency formula**

\[
E=\frac{\text{Row total}\times\text{Column total}}{\text{Grand total}}
\]

**Chi-square formula**

\[
\chi^2=\sum\frac{(O-E)^2}{E}
\]

where \(O\) is the observed frequency and \(E\) is the expected frequency.

**Degrees of freedom**

\[
df=(\text{Number of rows}-1)(\text{Number of columns}-1)
\]

The observed and expected counts, chi-square statistic and degrees of freedom are calculated using loops. The statistic is compared with the 5% chi-square critical-value table.

**Decision rule:** If the calculated chi-square value is greater than the critical value, reject \(H_0\). Otherwise, do not reject \(H_0\).

> The chi-square test requires expected cell frequencies to be sufficiently large. The code checks whether any expected frequency is below 5 and displays a warning if so.

In [ ]:
# Chi-square test of independence calculated manually.
# Rows = branch; columns = placement status.

chi_data = known_placement[["branch", "placed"]].dropna().copy()

if len(chi_data) == 0:
    print("No complete branch and placement records are available.")
else:
    observed_table = pd.crosstab(chi_data["branch"], chi_data["placed"])

    # Keep only the two expected placement columns if both are present.
    observed_table = observed_table.reindex(columns=["Yes", "No"], fill_value=0)

    row_count = observed_table.shape[0]
    column_count = observed_table.shape[1]

    if row_count < 2 or column_count < 2:
        print("At least two branches and both placement outcomes are required.")
    else:
        print("Observed frequency table:")
        display(observed_table)

        row_totals = []
        for i in range(row_count):
            row_total = 0
            for j in range(column_count):
                row_total = row_total + int(observed_table.iloc[i, j])
            row_totals.append(row_total)

        column_totals = []
        for j in range(column_count):
            column_total = 0
            for i in range(row_count):
                column_total = column_total + int(observed_table.iloc[i, j])
            column_totals.append(column_total)

        grand_total = 0
        for value in row_totals:
            grand_total = grand_total + value

        chi_square_value = 0
        expected_table = []
        expected_below_5 = False

        for i in range(row_count):
            expected_row = []
            for j in range(column_count):
                expected = (row_totals[i] * column_totals[j]) / grand_total
                observed = int(observed_table.iloc[i, j])

                expected_row.append(expected)
                if expected < 5:
                    expected_below_5 = True

                chi_square_value = chi_square_value + ((observed - expected) ** 2) / expected

            expected_table.append(expected_row)

        degrees_freedom_chi = (row_count - 1) * (column_count - 1)

        # 5% significance-level chi-square critical values.
        # For an unlisted df, use the next lower listed df.
        chi_table = {
            1: 3.841, 2: 5.991, 3: 7.815, 4: 9.488, 5: 11.070,
            6: 12.592, 7: 14.067, 8: 15.507, 9: 16.919, 10: 18.307,
            12: 21.026, 15: 24.996, 20: 31.410, 24: 36.415, 30: 43.773
        }

        chi_table_df = 1
        for df_value in chi_table:
            if df_value <= degrees_freedom_chi and df_value > chi_table_df:
                chi_table_df = df_value

        critical_chi = chi_table[chi_table_df]

        expected_df = pd.DataFrame(
            expected_table,
            index=observed_table.index,
            columns=observed_table.columns
        )

        print("\\nExpected frequency table:")
        display(expected_df.round(2))
        print("Grand total:", grand_total)
        print("Degrees of freedom:", degrees_freedom_chi)
        print("Calculated chi-square value:", round(chi_square_value, 3))
        print("Critical chi-square value (5%):", critical_chi)
        print("Table degrees of freedom used:", chi_table_df)

        if expected_below_5:
            print("WARNING: At least one expected frequency is below 5.")
            print("The chi-square approximation may not be reliable. Combine sparse branch groups only if there is a valid academic reason, or use an appropriate alternative test.")
        else:
            print("Expected-frequency check: All expected frequencies are at least 5.")

        if chi_square_value > critical_chi:
            print("Decision: Reject H0.")
            print("Conclusion: Branch and placement status show a statistically significant association in this dataset.")
        else:
            print("Decision: Do not reject H0.")
            print("Conclusion: The data does not provide sufficient evidence of an association between branch and placement status.")

        print("Note: This test identifies association, not cause and effect.")

## 21. Logistic Regression: Predicting Placement Status

Logistic Regression is a **classification algorithm** used when the outcome has two possible categories.

In this project, we use:

- **Input (X):** CGPA
- **Output (Y):** Placement status
- Placed = 1
- Not Placed = 0

We use only CGPA as the input so that the model remains easy to understand and explain.

**Logistic Regression formula**

\[
P(Y=1)=\frac{1}{1+e^{-(b_0+b_1X)}}
\]

The formula converts the linear expression into a probability between 0 and 1.

**Classification rule**

- If predicted probability is 0.5 or more, predict Placed (1).
- If predicted probability is below 0.5, predict Not Placed (0).

We will divide the available records into training and testing data. The model learns from the training data and is evaluated on the testing data.

> This is an educational model based on a mock dataset. It should not be used to make actual placement decisions.

In [ ]:
# Import the basic tools required for Logistic Regression.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

print("Logistic Regression tools imported successfully.")

### 21.1 Prepare the data

We select only CGPA and placement status. Rows with missing values in either column are removed for this model.

The placement labels are converted into numbers:

- Yes becomes 1
- No becomes 0

In [ ]:
# Select the input and output columns.
logistic_data = data[["CGPA", "placed"]].copy()

# Remove rows where CGPA or placement status is missing.
logistic_data = logistic_data.dropna()

# Convert placement status into numbers.
logistic_data["placed_number"] = logistic_data["placed"].map({
    "Yes": 1,
    "No": 0
})

# Remove any rows with labels other than Yes or No.
logistic_data = logistic_data.dropna(subset=["placed_number"])

X = logistic_data[["CGPA"]]
y = logistic_data["placed_number"].astype(int)

print("Total records used:", len(logistic_data))
print("Placed records:", (y == 1).sum())
print("Not-placed records:", (y == 0).sum())

### 21.2 Split the data into training and testing sets

The training data is used to teach the model. The testing data is kept separate to check how the model performs on records it did not train on.

We use 80% of the data for training and 20% for testing.

In [ ]:
# Split the dataset into training and testing portions.
if len(logistic_data) < 10 or y.nunique() < 2 or y.value_counts().min() < 2:
    print("There are not enough records in both placement groups for a reliable train-test split.")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    print("Training records:", len(X_train))
    print("Testing records:", len(X_test))

### 21.3 Train the Logistic Regression model

The Logistic Regression algorithm learns the relationship between CGPA and the probability of being placed.

In [ ]:
if len(logistic_data) >= 10 and y.nunique() == 2 and y.value_counts().min() >= 2:
    # Create the Logistic Regression model.
    logistic_model = LogisticRegression()

    # Train the model.
    logistic_model.fit(X_train, y_train)

    print("Logistic Regression model trained successfully.")
    print("Model intercept:", round(logistic_model.intercept_[0], 4))
    print("CGPA coefficient:", round(logistic_model.coef_[0][0], 4))

### 21.4 Make predictions and evaluate the model

The model predicts placement outcomes for the testing records.

**Accuracy formula**

\[
Accuracy=\frac{\text{Number of correct predictions}}
{\text{Total number of predictions}}
\]

A confusion matrix compares actual outcomes with predicted outcomes.

In [ ]:
if len(logistic_data) >= 10 and y.nunique() == 2 and y.value_counts().min() >= 2:
    # Predict placement status for the test data.
    y_pred = logistic_model.predict(X_test)

    # Calculate accuracy.
    model_accuracy = accuracy_score(y_test, y_pred)

    # Create confusion matrix.
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    print("Model Accuracy:", round(model_accuracy * 100, 2), "%")
    print("\\nConfusion Matrix:")
    print("Rows = Actual values; Columns = Predicted values")
    print("                 Predicted Not Placed   Predicted Placed")
    print("Actual Not Placed       ", cm[0][0], "                 ", cm[0][1])
    print("Actual Placed           ", cm[1][0], "                 ", cm[1][1])

    print("\\nCorrect predictions:", cm[0][0] + cm[1][1])
    print("Incorrect predictions:", cm[0][1] + cm[1][0])

### 21.5 Visualise the Logistic Regression curve

The curve shows how the predicted probability of placement changes with CGPA. The horizontal line at 0.5 represents the classification threshold.

In [ ]:
if len(logistic_data) >= 10 and y.nunique() == 2 and y.value_counts().min() >= 2:
    # Generate CGPA values for drawing a smooth curve.
    cgpa_values = pd.DataFrame({
        "CGPA": [
            min(X["CGPA"]) + i * (max(X["CGPA"]) - min(X["CGPA"])) / 100
            for i in range(101)
        ]
    })

    # Calculate probability of being placed for each CGPA value.
    placement_probabilities = logistic_model.predict_proba(cgpa_values)[:, 1]

    plt.figure(figsize=(8, 5))
    plt.scatter(
        X_test["CGPA"],
        y_test,
        label="Actual test outcomes",
        alpha=0.7
    )
    plt.plot(
        cgpa_values["CGPA"],
        placement_probabilities,
        label="Predicted probability of placement"
    )
    plt.axhline(0.5, linestyle="--", label="0.5 decision threshold")
    plt.xlabel("CGPA")
    plt.ylabel("Placement outcome / probability")
    plt.title("Logistic Regression: CGPA and Placement")
    plt.legend()
    plt.show()

### 21.6 Interpretation and limitations

- Accuracy tells us the percentage of test records classified correctly.
- The confusion matrix shows the number of correct and incorrect predictions for each placement group.
- The sigmoid curve shows the estimated probability of placement for different CGPA values.
- This model uses only CGPA. Other factors such as aptitude, projects, internships and communication skills are not included.
- Accuracy alone may be misleading if the number of placed and not-placed students is very different.
- The dataset is mock data, so the model's results should not be generalised to real placement outcomes.

**Important:** Logistic Regression identifies patterns in the available data. It does not prove that CGPA causes a student to get placed.

## 22. Simple Linear Regression: CGPA and Package

Simple linear regression studies the relationship between one independent variable and one dependent variable.

- **Independent variable (X):** CGPA
- **Dependent variable (Y):** Package in LPA

The regression equation is:

\[
\hat{Y}=b_0+b_1X
\]

We calculate the slope and intercept manually:

\[
b_1=\frac{\sum(X-\bar X)(Y-\bar Y)}{\sum(X-\bar X)^2}
\]

\[
b_0=\bar Y-b_1\bar X
\]

Here, \(b_0\) is the intercept and \(b_1\) is the slope. We use only placed students with both CGPA and package values available.

The slope describes the estimated change in package for a one-point increase in CGPA in this fitted sample. It does not mean CGPA alone determines a student's package.

In [ ]:
# Simple Linear Regression using mathematical formulas.
# Slope b1 = Sum[(X - mean_X)(Y - mean_Y)] / Sum[(X - mean_X)^2]
# Intercept b0 = mean_Y - b1 * mean_X

regression_data = placed_students[["CGPA", "Package (LPA)"]].dropna().copy()

X = []
Y = []

for row_index in range(len(regression_data)):
    X.append(float(regression_data.iloc[row_index]["CGPA"]))
    Y.append(float(regression_data.iloc[row_index]["Package (LPA)"]))

n = len(X)
print("Placed students used for regression:", n)

if n < 2:
    print("At least two complete placed-student records are required.")
else:
    # Calculate means manually.
    total_x = 0
    total_y = 0

    for i in range(n):
        total_x = total_x + X[i]
        total_y = total_y + Y[i]

    x_mean = total_x / n
    y_mean = total_y / n

    numerator = 0
    denominator = 0

    for i in range(n):
        numerator = numerator + ((X[i] - x_mean) * (Y[i] - y_mean))
        denominator = denominator + ((X[i] - x_mean) ** 2)

    if denominator == 0:
        print("Regression cannot be calculated because all CGPA values are identical.")
    else:
        slope = numerator / denominator
        intercept = y_mean - (slope * x_mean)

        print("Mean of CGPA (X):", round(x_mean, 3))
        print("Mean of Package (Y):", round(y_mean, 3))
        print("Numerator:", round(numerator, 3))
        print("Denominator:", round(denominator, 3))
        print("Slope (b1):", round(slope, 3))
        print("Intercept (b0):", round(intercept, 3))
        print("Regression equation: Estimated Package =",
              round(intercept, 3), "+", round(slope, 3), "* CGPA")

## 23. Plot the Regression Line

The scatterplot displays the observed CGPA and package values. The straight line represents the fitted simple linear regression model.

In [ ]:
if n >= 2 and denominator != 0:
    predicted_package = []

    for i in range(n):
        predicted_value = intercept + (slope * X[i])
        predicted_package.append(predicted_value)

    # Sort the points by CGPA so the regression line is drawn neatly.
    sorted_pairs = []
    for i in range(n):
        sorted_pairs.append((X[i], predicted_package[i]))
    sorted_pairs.sort()

    sorted_x = []
    sorted_y = []
    for pair in sorted_pairs:
        sorted_x.append(pair[0])
        sorted_y.append(pair[1])

    plt.scatter(X, Y, label="Actual package")
    plt.plot(sorted_x, sorted_y, color="red", label="Regression line")
    plt.title("Simple Linear Regression: CGPA vs Package")
    plt.xlabel("CGPA")
    plt.ylabel("Package (LPA)")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 24. Calculate R-squared and Make an Estimate

R-squared is calculated manually using:

\[
R^2=1-\frac{\sum(Y-\hat Y)^2}{\sum(Y-\bar Y)^2}
\]

It indicates the proportion of variation in package values explained by the fitted straight-line relationship in this sample.

We also use the regression equation to estimate a package for a sample CGPA value. This is an illustrative estimate, not a salary promise or placement guarantee.

In [ ]:
# R-squared calculated manually.
# R^2 = 1 - (Sum of squared residuals / Sum of squared total deviations)

if n >= 2 and denominator != 0:
    predicted_values = []
    ss_total = 0
    ss_residual = 0

    for i in range(n):
        predicted = intercept + (slope * X[i])
        predicted_values.append(predicted)

        ss_total = ss_total + (Y[i] - y_mean) ** 2
        ss_residual = ss_residual + (Y[i] - predicted) ** 2

    if ss_total == 0:
        print("R-squared cannot be calculated because package values do not vary.")
    else:
        r_squared = 1 - (ss_residual / ss_total)
        print("Sum of squared total deviations:", round(ss_total, 3))
        print("Sum of squared residuals:", round(ss_residual, 3))
        print("R-squared:", round(r_squared, 3))
        print("Variation explained by the fitted line:",
              round(r_squared * 100, 1), "%")

    # Use the regression equation for an example estimate.
    student_cgpa = 8.0
    estimated_package = intercept + (slope * student_cgpa)

    print("\\nExample CGPA:", student_cgpa)
    print("Estimated package (LPA):", round(estimated_package, 2))
    print("This is an illustrative estimate based only on this dataset.")

## 25. Key Findings

Complete this section after running all cells with the project CSV. Use the actual values produced by your notebook.

1. **Placement distribution:** Report the number and percentage of placed and not-placed students among records with known status.
2. **Branch comparison:** Identify branches with higher and lower placement percentages. Consider the number of students in each branch.
3. **CGPA:** Describe the mean, median and standard deviation.
4. **Aptitude:** Describe any visible differences between the two placement groups.
5. **Correlation:** Identify one or two relationships from the correlation matrix and explain their direction and strength.
6. **Independent-samples t-test:** Report the t-statistic, degrees of freedom and decision at the 5% significance level.
7. **Chi-square test:** Report the chi-square statistic, degrees of freedom and decision about branch and placement status.
8. **Logistic Regression:** Report test accuracy and explain the confusion matrix.
9. **Linear Regression:** Report the slope and R-squared, then explain what they mean in simple language.

Do not write a finding unless it is supported by the output from your own run.

## 26. Limitations

- The dataset is mock data and may not represent actual placement conditions.
- Some records contain missing or inconsistent values.
- Branch-wise results can be affected by unequal branch sizes.
- Correlation does not establish causation.
- Simple linear regression uses only one independent variable.
- Package information is available only for placed students, so regression is restricted to that group.
- The model should not be used to make real student selection or salary decisions.
- Hypothesis-test conclusions apply only to this mock dataset; assumptions such as independent observations and adequate expected frequencies must be considered.
- Logistic Regression uses CGPA as its only input, so it does not capture all factors related to placement.

## 27. Conclusion

This project demonstrates a basic data science workflow using Python. The dataset was inspected and cleaned, numerical summaries and probabilities were calculated, placement patterns were visualised, relationships between selected variables were explored using covariance and correlation, and two hypothesis tests were performed. Finally, simple linear regression was applied to study the relationship between CGPA and package among placed students.

The project helped apply beginner-level concepts of Python, Pandas, descriptive statistics, probability, visualisation and simple linear regression to a practical dataset. The results should be interpreted only within the limitations of the mock data.

## 28. Save the Cleaned Dataset

The cleaned data is exported to a new CSV file so that it can be reviewed or used in future exercises.

In [ ]:
cleaned_file_name = "placement_cleaned_by_notebook.csv"
data.to_csv(cleaned_file_name, index=False)
print("Cleaned dataset saved as:", cleaned_file_name)

## 29. Concepts Demonstrated

| FDS concept | Notebook section |
|---|---|
| CSV data handling | 3–4 |
| Data quality checks | 5 |
| Data cleaning and preprocessing | 6–7 |
| Descriptive statistics | 9 |
| Basic probability | 10 |
| Frequency distribution | 11 |
| Data visualisation | 12–16 |
| Covariance and correlation | 17–18 |
| Independent-samples t-test | 19 |
| Chi-square test of independence | 20 |
| Logistic Regression | 21 |
| Simple linear regression | 22–24 |
| Interpretation and limitations | 25–27 |
| Exporting cleaned data | 28 |

**End of project notebook.**